# 00 — The group and its data

**Goal (PROBLEM.md):** a multi-site hospital group — branches, regional boards, departments —
must decide where to add staff. This notebook lays out *who is in the group* and builds the
*operational panel* every later decision reads.

- Branch hospital ↔ NHS trust (35, top-5 per region, rule-derived; audit: notebooks/00 & a100/01, **PASSED**)
- Board ↔ ICB · Region ↔ NHS England region (7) · Department ↔ specialty (16–23 per branch)
- Demand = Part_3 new referrals · Backlog = Part_2 waiting list · Capacity delivered = Part_1A+1B

In [1]:
import json
import glob
import os
import pandas as pd

# the org chart: regions -> branches (with board + departments + typical demand)
net = json.load(open("../config/nhs_network.json"))
rows = [{"region": r["region"], "branch": t["code"], "name": t["name"].title()[:44],
         "board_icb": t["icb"], "departments": t["specialties"],
         "demand_per_month": t["mean_monthly_referrals"]}
        for r in net["regions"] for t in r["trusts"]]
org = pd.DataFrame(rows)
print(f"group: {org['branch'].nunique()} branches | {org['region'].nunique()} regions | "
      f"{org['board_icb'].nunique()} boards | ~{org['demand_per_month'].sum():,} referrals/month")
org.head(10)

group: 35 branches | 7 regions | 29 boards | ~675,458 referrals/month


,region,branch,name,board_icb,departments,demand_per_month
0,East of England,RAJ,Mid And South Essex Nhs Foundation Trust,QH8,23,39885
1,East of England,RDE,East Suffolk And North Essex Nhs Foundation,QJG,23,18576
2,East of England,RC9,Bedfordshire Hospitals Nhs Foundation Trust,QHG,22,18532
3,East of England,RM1,Norfolk And Norwich University Hospitals Nhs,QMM,21,16025
4,East of England,RGT,Cambridge University Hospitals Nhs Foundatio,QUE,22,14612
5,London,RJ1,Guy'S And St Thomas' Nhs Foundation Trust,QKK,23,27460
6,London,RJZ,King'S College Hospital Nhs Foundation Trust,QKK,23,23569
7,London,R1H,Barts Health Nhs Trust,QMF,23,23531
8,London,RYJ,Imperial College Healthcare Nhs Trust,QRV,22,22340
9,London,RAL,Royal Free London Nhs Foundation Trust,QMJ,23,21821


In [2]:
# operational panel: build data/processed/rtt_long.csv from raw if absent (deterministic; ~8 min)
OUT_CSV = "../data/processed/rtt_long.csv"
if os.path.exists(OUT_CSV):
    long = pd.read_csv(OUT_CSV)
    print(f"panel already built - loaded {len(long)} rows")
else:
    ID_COLS = ["Period", "Provider Org Code", "Provider Org Name", "RTT Part Type",
               "Treatment Function Code", "Treatment Function Name"]
    BAND_COLS = [f"Gt {i:02d} To {i + 1:02d} Weeks SUM 1" for i in range(104)] + ["Gt 104 Weeks SUM 1"]
    TOTAL_COLS = ["Total", "Total All"]  # counts live in Total All (Total empty for Part_2/2A/3)
    codes = set(org["branch"])
    monthly = []
    for path in sorted(glob.glob("../data/nhs_rtt_referral_to_treatment/rtt_*.csv")):
        month = os.path.basename(path)[4:11]
        df = pd.read_csv(path, usecols=ID_COLS + BAND_COLS + TOTAL_COLS, dtype={c: str for c in ID_COLS})
        df = df[df["Treatment Function Code"] != "C_999"]           # drop double-counting totals
        df = df[df["Provider Org Code"].isin(codes)]                 # network branches only
        df[BAND_COLS + TOTAL_COLS] = df[BAND_COLS + TOTAL_COLS].apply(pd.to_numeric, errors="coerce").fillna(0)
        df = df.assign(weeks_0_18=df[BAND_COLS[:18]].sum(axis=1),
                       weeks_18_52=df[BAND_COLS[18:52]].sum(axis=1),
                       weeks_52_plus=df[BAND_COLS[52:]].sum(axis=1))
        out = df.groupby(["Provider Org Code", "Provider Org Name", "RTT Part Type",
                          "Treatment Function Code", "Treatment Function Name"], as_index=False)[
            ["weeks_0_18", "weeks_18_52", "weeks_52_plus", "Total", "Total All"]].sum()
        out.insert(0, "month", month)
        out.columns = ["month", "provider_code", "provider_name", "part", "specialty_code",
                       "specialty_name", "weeks_0_18", "weeks_18_52", "weeks_52_plus", "total", "total_all"]
        monthly.append(out)
        print(month, end=" ", flush=True)
    long = pd.concat(monthly, ignore_index=True)
    os.makedirs(os.path.dirname(OUT_CSV), exist_ok=True)
    long.to_csv(OUT_CSV, index=False)
    print(f"\nbuilt {OUT_CSV}: {len(long)} rows")

panel already built - loaded 210328 rows


In [3]:
# panel checks: complete, consistent with the org chart
assert long["provider_code"].nunique() == 35 and set(long["provider_code"]) == set(org["branch"])
presence = long.groupby("provider_code")["month"].nunique()
assert (presence == long["month"].nunique()).all(), "a branch is missing months"
assert set(long["part"]) == {"Part_1A", "Part_1B", "Part_2", "Part_2A", "Part_3"}
print(f"panel OK: {long['month'].nunique()} months ({long['month'].min()}..{long['month'].max()}) | "
      f"35 branches | {long['specialty_code'].nunique()} departments | {len(long):,} rows")

panel OK: 60 months (2021-04..2026-03) | 35 branches | 23 departments | 210,328 rows
